# 08 – Evaluation: Matriz de Confusión

**Proyecto:** Encuesta Permanente de Empleo Nacional (EPEN)  
**Objetivo:** Analizar en detalle la matriz de confusión del modelo final, interpretando los tipos de error (falsos positivos y falsos negativos) en el contexto del problema de desempleo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import joblib
import os

SEL_DIR = os.path.join('..', 'data', 'selected')
SPLIT_DIR = os.path.join('..', 'data', 'split')
MODEL_DIR = os.path.join('..', 'models')
TARGET = 'target_desocupado'

try:
    df_train = pd.read_csv(os.path.join(SEL_DIR, 'epen_selected.csv'))
    X_test = pd.read_csv(os.path.join(SPLIT_DIR, 'X_test.csv'))
    y_test = pd.read_csv(os.path.join(SPLIT_DIR, 'y_test.csv')).squeeze()
    selected = pd.read_csv(os.path.join(SEL_DIR, 'selected_features.csv'))['selected_feature'].tolist()
    X_train = df_train[[c for c in selected if c in df_train.columns]]
    y_train = df_train[TARGET]
    X_test = X_test[[c for c in selected if c in X_test.columns]].fillna(0)
except FileNotFoundError:
    np.random.seed(42)
    n_train, n_test, n_feat = 800, 200, 8
    X_train = pd.DataFrame(np.random.randn(n_train, n_feat), columns=[f'f{i}' for i in range(n_feat)])
    y_train = pd.Series(np.random.choice([0, 1], n_train, p=[0.50, 0.50]))
    X_test = pd.DataFrame(np.random.randn(n_test, n_feat), columns=[f'f{i}' for i in range(n_feat)])
    y_test = pd.Series(np.random.choice([0, 1], n_test, p=[0.50, 0.50]))

rf_path = os.path.join(MODEL_DIR, 'random_forest.pkl')
if os.path.exists(rf_path):
    model = joblib.load(rf_path)
else:
    model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('Predicciones listas.')

## 1. Matriz de Confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f'Verdaderos Negativos  (TN): {tn:>5} – No desocupados correctamente clasificados')
print(f'Falsos Positivos      (FP): {fp:>5} – No desocupados clasificados como desocupados')
print(f'Falsos Negativos      (FN): {fn:>5} – Desocupados clasificados como NO desocupados ⚠️')
print(f'Verdaderos Positivos  (TP): {tp:>5} – Desocupados correctamente identificados')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión – conteos
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['No desocupado', 'Desocupado'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusión – Conteos')

# Matriz de confusión – normalizada
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['No desocupado', 'Desocupado'],
            yticklabels=['No desocupado', 'Desocupado'])
axes[1].set_title('Matriz de Confusión – Normalizada')
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

## 2. Interpretación en contexto

| Error | Tipo | Impacto en política pública |
|---|---|---|
| **Falso Negativo (FN)** | Desocupado → No desocupado | **Alto**: Personas en desempleo no reciben apoyo |
| **Falso Positivo (FP)** | No desocupado → Desocupado | **Medio**: Recursos asignados innecesariamente |

> En este contexto, **minimizar los Falsos Negativos** (mejorar el Recall) es prioritario para no dejar a personas desocupadas sin identificar.

In [ ]:
# Métricas derivadas de la matriz de confusión
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0  # Precision
npv = tn / (tn + fn) if (tn + fn) > 0 else 0  # Negative Predictive Value

print(f'Sensibilidad (Recall) : {sensitivity:.4f}')
print(f'Especificidad         : {specificity:.4f}')
print(f'Precisión (PPV)       : {ppv:.4f}')
print(f'NPV                   : {npv:.4f}')